In [ ]:
import piplite
await piplite.install("ehyd-tools")

# Bemessung einer Flächenversickerung

$$ A_S = \frac{A_{Bem}}{\frac{10^7 k_i}{r_{D,n}}-1} $$

mit:
- $A_S$ = Erforderliche Sickerflächen in m²
- $A_{Bem}$ = Netto angeschlossene Fläche in m²
- $k_i$ = Bemessungsrelevante Infiltrationsrate in m/s
- $r_{D,n}$ = Regenspende Dauerstufe $D$ und Wiederkehrzeit $T_n$ in l/(s*ha)

## Implementierung als Funktion:

In [22]:
def sickerflaeche(rdn, abem, ki):
    return abem / ((1e7 * ki / rdn) - 1)

## Importiere Regenereignisse:

In [13]:
from ehyd_tools.design_rainfall import read_ehyd_design_rainfall, get_calculation_method

df_ehyd = read_ehyd_design_rainfall(r"data/Bemessungsniederschlag_2020_Gitterpunkt_5107.txt")
df_hdn = get_calculation_method(df_ehyd)
df_hdn.head()

return period,1,2,3,5,10,20,25,30,50,75,100
duration,,,,,,,,,,,
5,8.0,9.9,11.0,12.4,14.4,16.7,17.5,18.1,19.9,21.3,22.2
10,13.2,16.0,17.9,21.0,25.1,29.3,30.6,31.7,34.8,37.2,38.9
15,16.7,20.1,22.8,26.7,32.1,37.4,39.1,40.5,44.4,47.5,49.8
20,19.1,23.1,26.2,30.6,36.6,42.8,44.7,46.3,50.8,54.4,56.9
30,22.4,27.3,30.9,36.3,43.4,50.7,53.0,54.9,60.2,64.4,67.5


## Bemessungsbeispiel:

### Parameter
- $A_{Bem}$ = 1000 m²
- $k_i$ = $10^{-5}$ m/s
- $T_n$ = 30 a

In [10]:
a_bem = 1000
k_i = 1e-5

### Berechnung der erforderlichen Sickerfläche für jede Dauerstufe
Umrechnen von $h_{D,n}$ in $r_{D,n}$

In [18]:
s_rdn = df_hdn[30] * 166.7 / df_hdn.index
s_rdn = s_rdn.rename("Regenspende l/(s*ha)")
s_rdn.head()

duration
5     603.4540
10    528.4390
15    450.0900
20    385.9105
30    305.0610
Name: Regenspende l/(s*ha), dtype: float64

Berechnen der erforderlichen Sickerfläche für jede Dauerstufe

In [25]:
a_erf = s_rdn.apply(sickerflaeche, args=(a_bem, k_i)).rename("A_erf")
a_erf.head()

duration
5    -1198.627879
10   -1233.405456
15   -1285.640835
20   -1349.759802
30   -1487.659770
Name: A_erf, dtype: float64

## Maßgebende Sickerfläche = maximale Sickerfläche

In [27]:
A_erf = a_erf.max()
print(f"Die notwendige Sickerfläche um das maßgebende 30-jährliche Ereignis zu Versickern beträgt {A_erf:.0f} m²")

Die notwendige Sickerfläche um das maßgebende 30-jährliche Ereignis zu Versickern beträgt 3522 m²


# Übung: Bemiss das notwendige Sickervolumen für eine Sickermulde

$$ V_M = \left[ 10^{-7} r_{D,n} (A_{Bem} + A_M) - A_{S,m} k_i \right] * 60 D f_Z $$

mit:
- $A_{S,m}$ = Mittlere Sickerfläche der Mulde [m2]
- $A_{Bem}$ = Netto angeschlossene Fläche [m2]
- $k_i$ = Bemessungsrelevante Infiltransrate [m/s]
- $r_{D,n}$ = Regenspende Dauerstufe $D$ und Wiederkehrzeit $T_n$  in [l/(s*ha)]
- $V_M$ = erforderliches Speichervolumen der Mulde [m3]
- $A_M$ = Überregnete Fläche der Mulde [m2]
- $f_Z$ = Zuschlagsfaktor [-]
- $D$ = Dauerstufe D des Bemessungsregens [min]

### Parameter
- $A_{Bem}$ = 1000 m²
- $k_i$ = $10^{-5}$ m/s
- $T_n$ = 30 a
- $A_S = A_M$ = 200 m²
- $f_Z$ = 1.0


**F1:** Wie tief wird das Becken eingestaut?

**F2:** Wie lange benötigt das Becken für die Entleerung?